In [85]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import re
from itertools import combinations
from scipy.stats import spearmanr, pearsonr



## Correlación de Spearman

In [130]:
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        df = pd.read_csv(ruta_completa)
        df = df.pivot(index='Date', columns='Buoy', values='Chl') # pivot para tener una columna para cada boya
        dfs_boyas[nombre_sin_extension] = df

In [142]:
df = dfs_boyas["df_boyas_imida_depth_eq_1"]
resultados = []

# Iterar sobre todos los pares únicos de boyas
for b1, b2 in combinations(df.columns, 2):
    serie1 = df[b1]
    serie2 = df[b2]

    # Eliminar valores NaN simultáneamente
    datos = pd.concat([serie1, serie2], axis=1).dropna()
    if datos.empty:
        resultados.append({
            'Buoy1': b1, 'Buoy2': b2,
            'Spearman_rho': None, 'Spearman_pval': None,
            'Pearson_r': None, 'Pearson_pval': None
        })
        continue

    # Calcular Spearman
    rho_s, pval_s = spearmanr(datos.iloc[:, 0], datos.iloc[:, 1])

    # Calcular Pearson
    r_p, pval_p = pearsonr(datos.iloc[:, 0], datos.iloc[:, 1])

    resultados.append({
        'Buoy1': b1, 'Buoy2': b2,
        'Spearman_rho': rho_s, 'Spearman_pval': pval_s,
        'Pearson_r': r_p, 'Pearson_pval': pval_p
    })

# Crear el DataFrame final
df_corr = pd.DataFrame(resultados)
df_corr = df_corr.sort_values(by='Spearman_rho', ascending=False)

In [143]:
# Especifica la boya de interés
boya_objetivo = 'CTD9'

# Filtrar todas las filas donde la boya aparezca como Buoy1 o Buoy2
filtro = (df_corr['Buoy1'] == boya_objetivo) | (df_corr['Buoy2'] == boya_objetivo)
df_filtrado = df_corr[filtro].copy()

# Crear una columna con la "otra boya"
# df_filtrado['Otra_boya'] = df_filtrado.apply(
#     lambda row: row['Buoy2'] if row['Buoy1'] == boya_objetivo else row['Buoy1'],
#     axis=1
# )

# Ordenar por Spearman rho descendente y mostrar las 4 más altas
df_top4 = df_filtrado.sort_values(by='Spearman_rho', ascending=False).head(4)

df_top4

,Buoy1,Buoy2,Spearman_rho,Spearman_pval,Pearson_r,Pearson_pval
18,CTD10,CTD9,0.759523,1.794539e-65,0.771572,9.040033e-69
44,CTD3,CTD9,0.739157,3.805817e-60,0.731157,2.946917e-58
54,CTD8,CTD9,0.731494,2.462223e-58,0.706749,6.852141e-53
33,CTD12,CTD9,0.723203,8.454879e-56,0.703569,1.231500e-51


## Clustering de series temporales con tslearn

In [90]:
from tslearn.utils import to_time_series, to_time_series_dataset
from tslearn.clustering import TimeSeriesKMeans
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter
from itertools import combinations

In [112]:
# CTD9 a parte porque solamente aparece en los conjuntos de datos de profundidades <1m 
for df_name, df in dfs_boyas.items():
    try:
        dfs_boyas[df_name] = df.drop(columns=["CTD9"])
    except:
        continue

In [122]:
agrupaciones = {}

for df_name, df in dfs_boyas.items():
    #dataset = to_time_series_dataset([df[col] for col in df.columns])
    scaler = StandardScaler()
    df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
    df = df.fillna(method="ffill").fillna(method="bfill")  # si hay pocos NaNs
    dataset = to_time_series_dataset([df[col] for col in df.columns])
    km = TimeSeriesKMeans(n_clusters=4, metric="dtw", random_state=0)
    km.fit(dataset)

    # Agrupar boyas por clúster
    agrupadas = defaultdict(list)
    for buoy, label in zip(df.columns, km.labels_):
        agrupadas[label].append(buoy)

    # Mostrar agrupaciones por línea
    for cluster_id in sorted(agrupadas):
        #print(f"Cluster {cluster_id}: {', '.join(agrupadas[cluster_id])}")
        agrupaciones[f"{df_name}"] = agrupadas


In [123]:
agrupaciones_todas = [(nombre, grupos) for nombre, grupos in agrupaciones.items()]

coincidencias = Counter()

for nombre, agrupacion in agrupaciones_todas:
    for grupo in agrupacion.values():
        for b1, b2 in combinations(sorted(grupo), 2):
            coincidencias[(b1, b2)] += 1

# Convertir a DataFrame para visualizar
df_coincidencias = pd.DataFrame([
    {'Buoy1': b1, 'Buoy2': b2, 'Coincidencias': count}
    for (b1, b2), count in coincidencias.items()
])

df_coincidencias = df_coincidencias.sort_values(by='Coincidencias', ascending=False)


Resultados con 3 clusters:

In [125]:
df_coincidencias.loc[df_coincidencias["Coincidencias"]>4]

,Buoy1,Buoy2,Coincidencias
5,CTD2,CTD3,9
15,CTD12,CTD8,9
12,CTD10,CTD12,8
14,CTD10,CTD8,8
4,CTD12,CTD6,7
13,CTD10,CTD6,6
29,CTD1,CTD3,6
16,CTD6,CTD8,6
8,CTD3,CTD4,6
28,CTD1,CTD2,5


Resultados con 4 clusters

In [126]:
df_coincidencias.loc[df_coincidencias["Coincidencias"]>4]

,Buoy1,Buoy2,Coincidencias
5,CTD2,CTD3,9
15,CTD12,CTD8,9
12,CTD10,CTD12,8
14,CTD10,CTD8,8
4,CTD12,CTD6,7
13,CTD10,CTD6,6
29,CTD1,CTD3,6
16,CTD6,CTD8,6
8,CTD3,CTD4,6
28,CTD1,CTD2,5


In [38]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
df_scaled = df_scaled.fillna(method="ffill").fillna(method="bfill")  # si hay pocos NaNs

dataset = to_time_series_dataset([df_scaled[col] for col in df_scaled.columns])
km = TimeSeriesKMeans(n_clusters=3, metric="dtw", random_state=0)
km.fit(dataset)

TimeSeriesKMeans(metric='dtw', random_state=0)

In [104]:
labels = km.labels_

# Asociar cada boya (columna) con su cluster
asignaciones = pd.DataFrame({
    'Buoy': df.columns,
    'Cluster': labels
})

asignaciones

,Buoy,Cluster
0,CTD1,0
1,CTD10,2
2,CTD11,2
3,CTD12,1
4,CTD2,0
5,CTD3,0
6,CTD4,0
7,CTD6,2
8,CTD7,0
9,CTD8,1


In [39]:
labels = km.labels_

# Asociar cada boya (columna) con su cluster
asignaciones = pd.DataFrame({
    'Buoy': df.columns,
    'Cluster': labels
})

asignaciones

,Buoy,Cluster
0,CTD1,0
1,CTD10,2
2,CTD11,2
3,CTD12,1
4,CTD2,1
5,CTD3,0
6,CTD4,0
7,CTD6,2
8,CTD7,1
9,CTD8,1


In [41]:
print(np.bincount(km.labels_))


[3 4 3]
